# Section 5.3: a spline dictionary with two derivatives

The model is the one of Section 5.1: $\mathcal A f=\partial_x\big((1+\beta\cos x)\partial_xf\big)$ on the circle,
$\beta=1/2$, uniform invariant law, $T=1$, and the observable with $\hat f_k\propto(1+k^2)^{-7/2}$. The dictionary consists of
the $N$ periodic cubic B-splines $\psi_j(x)=\sum_{l\in\mathbb Z}\chi\big((x-jh)/h-lN\big)$, $0\le j<N$, on the uniform mesh of width $h=2\pi/N$, where $\chi$ is the cubic cardinal B-spline with support $[-2,2]$. These functions are
twice continuously differentiable and piecewise cubic, so $\mathcal V_N\subset\operatorname{dom}(\mathcal A)$, but
$\mathcal A\psi_j$ has a derivative with jumps at the mesh points and $\mathcal V_N\not\subset\operatorname{dom}(\mathcal A^2)$.
SDMD-1 and SDMD-R are defined as before. For SDMD-2 the notebook compares two substitutes for the third matrix
$\mathbf J_{ij}=\langle\psi_i,\mathcal A^2\psi_j\rangle$:

* the pointwise form, in which $\mathcal A^2\psi_j$ is evaluated by the differentiation formulas away from the mesh points
  (the fourth derivative of a cubic spline vanishes there);
* the energy form $\mathbf J^{\mathrm{en}}_{ij}=\langle\mathcal A\psi_i,\mathcal A\psi_j\rangle$ of Appendix A.2.

The two differ by the point masses of $\psi_j''''$ at the mesh points, which no sample of the invariant law sees.
The time step is $\Delta t=T/n$ with $n$ the smallest integer such that $\Delta t\,\|\mathcal A_N\|\le1.5$. The notebook reproduces
Table 8 (errors against $\mathcal K^Tf$ with exact matrices and with empirical matrices from $m$ i.i.d. uniform samples, seed 20260507).

Runtime: about two minutes.

In [1]:
import numpy as np
from scipy.linalg import eigh, expm

SEED = 20260507
beta, T = 0.5, 1.0

# reference solution K^T f in Fourier space (cutoff 256), as in Section 5.1
KREF = 256
ks = np.arange(-KREF, KREF + 1)
A_ref = np.zeros((ks.size, ks.size))
for i, k in enumerate(ks):
    A_ref[i, i] = -k**2
    if i + 1 < ks.size:
        A_ref[i + 1, i] = -beta / 2 * (k**2 + k)
    if i - 1 >= 0:
        A_ref[i - 1, i] = -beta / 2 * (k**2 - k)
f_hat = (1.0 + ks.astype(float)**2) ** (-3.5)
f_hat /= np.linalg.norm(f_hat)
d_ref, V_ref = np.linalg.eigh(A_ref)
u_hat = (V_ref * np.exp(T * d_ref)) @ (V_ref.T @ f_hat)


def fourier_values(coef, x):
    """Values at x of the real function with Fourier coefficients coef (even in k)."""
    return np.real(np.exp(1j * np.outer(x, ks)) @ coef)


def coefficient(x):
    """a, a', a'', a''' for a = 1 + beta cos x."""
    return 1.0 + beta * np.cos(x), -beta * np.sin(x), -beta * np.cos(x), beta * np.sin(x)


def bspline(t):
    """Cubic cardinal B-spline on [-2, 2] and its first three derivatives away from the knots."""
    s, sg = np.abs(t), np.sign(t)
    B = np.where(s < 1, 2 / 3 - s**2 + s**3 / 2, np.where(s < 2, (2 - s) ** 3 / 6, 0.0))
    B1 = np.where(s < 1, -2 * s + 1.5 * s**2, np.where(s < 2, -((2 - s) ** 2) / 2, 0.0)) * sg
    B2 = np.where(s < 1, -2 + 3 * s, np.where(s < 2, 2 - s, 0.0))
    B3 = np.where(s < 1, 3.0, np.where(s < 2, -1.0, 0.0)) * sg
    return B, B1, B2, B3


def arrays(x, N):
    """Evaluations of psi, A psi, and of the pointwise form of A^2 psi at the points x, as (len(x), N) arrays.

    A psi = a psi'' + a' psi' and, with g = A psi, A^2 psi = a g'' + a' g' with g' = a psi''' + 2 a' psi'' + a'' psi' and
    g'' = 3 a' psi''' + 3 a'' psi'' + a''' psi' away from the mesh points. Only four splines are nonzero at each point."""
    h = 2 * np.pi / N
    cell = np.floor(x / h).astype(int)
    s = x / h - cell                                  # local coordinate in [0, 1)
    a, a1, a2, a3 = coefficient(x)
    rows = np.arange(x.size)
    P, AP, A2P = (np.zeros((x.size, N)) for _ in range(3))
    for q in range(4):                                # splines j = cell - 1, ..., cell + 2
        B, B1, B2, B3 = bspline(s + 1 - q)
        f1, f2, f3 = B1 / h, B2 / h**2, B3 / h**3
        g1 = a * f3 + 2 * a1 * f2 + a2 * f1
        g2 = 3 * a1 * f3 + 3 * a2 * f2 + a3 * f1
        cols = (cell - 1 + q) % N
        P[rows, cols] = B
        AP[rows, cols] = a * f2 + a1 * f1
        A2P[rows, cols] = a * g2 + a1 * g1
    return P, AP, A2P


GL_X, GL_W = np.polynomial.legendre.leggauss(10)


def population(N):
    """Quadrature nodes and weights (10 Gauss-Legendre points per mesh cell) and the matrices G, H, J (pointwise), J (energy)."""
    h = 2 * np.pi / N
    xq = (np.arange(N)[:, None] * h + (GL_X[None, :] + 1) * h / 2).ravel()
    wq = np.tile(GL_W * h / 2, N) / (2 * np.pi)
    P, AP, A2P = arrays(xq, N)
    W = wq[:, None]
    return xq, wq, P, P.T @ (W * P), P.T @ (W * AP), P.T @ (W * A2P), AP.T @ (W * AP)


def step_count(norm_A):
    """Number of steps n and step dt = T/n, with n the smallest integer such that dt * norm_A <= 1.5."""
    n = int(np.ceil(T * norm_A / 1.5))
    return n, T / n


def propagators(A, B_pointwise, B_energy, dt, n):
    """e^{TA} and the n-th powers of the one-step matrices of SDMD-1, SDMD-R, SDMD-2 (pointwise), SDMD-2 (energy)."""
    I = np.eye(len(A))
    Q1 = I + dt * A
    maps = [Q1, Q1 + 0.5 * dt**2 * (A @ A), Q1 + 0.5 * dt**2 * B_pointwise, Q1 + 0.5 * dt**2 * B_energy]
    return [expm(T * A)] + [np.linalg.matrix_power(Q, n) for Q in maps], maps


NAMES = ["exponential", "SDMD-1", "SDMD-R", "SDMD-2 (pointwise)", "SDMD-2 (energy)"]


class Model:
    """Exact Galerkin matrices of the spline dictionary with N functions, the projected observable, and the error functional."""

    def __init__(self, N):
        self.N = N
        self.xq, self.wq, self.P, self.G, self.H, self.Jp, self.Je = population(N)
        self.A = np.linalg.solve(self.G, self.H)
        self.Bp = np.linalg.solve(self.G, self.Jp)
        self.Be = np.linalg.solve(self.G, self.Je)
        w, V = eigh(self.G)
        self.Gh, self.Ghi = (V * np.sqrt(w)) @ V.T, (V / np.sqrt(w)) @ V.T
        self.norm_A = float(eigh(-(self.H + self.H.T) / 2, self.G, eigvals_only=True).max())
        self.n, self.dt = step_count(self.norm_A)
        self.c0 = np.linalg.solve(self.G, self.P.T @ (self.wq * fourier_values(f_hat, self.xq)))    # Pi_N f
        self.truth = fourier_values(u_hat, self.xq)                                                   # K^T f

    def op_norm(self, M):
        """Operator norm on V_N for the coefficient matrix M."""
        return float(np.linalg.norm(self.Gh @ M @ self.Ghi, 2))

    def error(self, c):
        """L^2(rho) distance between sum_j c_j psi_j and K^T f."""
        return float(np.sqrt(np.sum(self.wq * (self.P @ c - self.truth) ** 2)))

## The pointwise form and the energy form of the third matrix

For $\psi_i,\psi_j$ in the dictionary, integration by parts on each mesh cell gives

$$\langle\mathcal A\psi_i,\mathcal A\psi_j\rangle=\int\psi_i\,(\mathcal A^2\psi_j)_{\mathrm{pointwise}}\,d\rho
+\frac1{2\pi}\sum_k\psi_i(x_k)\,a(x_k)^2\,[\psi_j'''](x_k),$$

where $x_k=kh$ are the mesh points and $[\psi_j''']$ is the jump of the third derivative, $h^{-3}(1,-4,6,-4,1)$ at the five
mesh points in the support of $\psi_j$. The cell checks this identity and the symmetry of $\mathbf H$.

In [2]:
for N in [16, 64]:
    M = Model(N)
    h = 2 * np.pi / N
    shift = lambda offsets_values: sum(v * np.roll(np.eye(N), o, axis=0) for o, v in offsets_values)   # [k, j] = v if k - j = o
    values = shift([(-1, 1 / 6), (0, 2 / 3), (1, 1 / 6)])                                             # psi_i(x_k)
    jumps = shift([(-2, 1.0), (-1, -4.0), (0, 6.0), (1, -4.0), (2, 1.0)]) / h**3                      # [psi_j'''](x_k)
    a_knots = coefficient(np.arange(N) * h)[0]
    J_jump = values.T @ (a_knots[:, None] ** 2 * jumps) / (2 * np.pi)
    print(f"N = {N:3d}: |J_energy - J_pointwise - J_jump| / |J_energy| = {np.abs(M.Je - M.Jp - J_jump).max() / np.abs(M.Je).max():.1e},"
          f"  |J_jump| / |J_energy| = {np.abs(J_jump).max() / np.abs(M.Je).max():.2f},"
          f"  |H - H^T| / |H| = {np.abs(M.H - M.H.T).max() / np.abs(M.H).max():.1e},  cond(G) = {np.linalg.cond(M.G):.1f}")

N =  16: |J_energy - J_pointwise - J_jump| / |J_energy| = 1.4e-15,  |J_jump| / |J_energy| = 1.04,  |H - H^T| / |H| = 3.2e-15,  cond(G) = 18.5
N =  64: |J_energy - J_pointwise - J_jump| / |J_energy| = 7.1e-15,  |J_jump| / |J_energy| = 1.00,  |H - H^T| / |H| = 1.7e-14,  cond(G) = 18.5


## Table 8, exact matrices

In [3]:
print("errors ||P Pi_N f - K^T f|| with exact matrices")
print("    N     n  dt|A_N|  Theta_N*dt   " + "  ".join(f"{name:>18s}" for name in NAMES) + "    norms of the four one-step maps")
exact = []
for N in [16, 32, 64, 128]:
    M = Model(N)
    props, maps = propagators(M.A, M.Bp, M.Be, M.dt, M.n)
    errs = [M.error(P @ M.c0) for P in props]
    theta = M.op_norm(M.Be - M.A @ M.A)
    exact.append(dict(N=N, n=M.n, err=errs))
    print(f"{N:5d} {M.n:5d}   {M.dt * M.norm_A:5.3f}   {theta * M.dt:8.2f}   " + "  ".join(f"{e:18.3e}" for e in errs)
          + "    " + " ".join(f"{M.op_norm(Q):.4f}" for Q in maps))
Ns = [r["N"] for r in exact]
for j, name in enumerate(NAMES):
    slope = np.polyfit(np.log(Ns[1:]), np.log([r["err"][j] for r in exact[1:]]), 1)[0]
    print(f"slope in N over N = 32, 64, 128, {name}: {slope:.2f}")
print("error divided by the error of the exponential:")
for r in exact:
    print(f"  N = {r['N']:3d}: " + ", ".join(f"{name} {r['err'][j] / r['err'][0]:.3g}" for j, name in enumerate(NAMES) if j))

errors ||P Pi_N f - K^T f|| with exact matrices
    N     n  dt|A_N|  Theta_N*dt          exponential              SDMD-1              SDMD-R  SDMD-2 (pointwise)     SDMD-2 (energy)    norms of the four one-step maps
   16    54   1.475       6.47            7.865e-06           3.866e-04           8.449e-06           5.426e-04           8.375e-06    1.0000 1.0000 1.0000 1.0000
   32   233   1.497      18.05            3.219e-07           8.914e-05           3.578e-07           1.239e-04           3.572e-07    1.0000 1.0000 1.0000 1.0000
   64   977   1.499      72.21            1.800e-08           2.124e-05           2.003e-08           2.945e-05           2.002e-08    1.0000 1.0000 1.0000 1.0000
  128  4002   1.500     298.63            1.093e-09           5.183e-06           1.212e-09           7.184e-06           1.212e-09    1.0000 1.0000 1.0000 1.0000
slope in N over N = 32, 64, 128, exponential: -4.10
slope in N over N = 32, 64, 128, SDMD-1: -2.05
slope in N over N = 32, 64, 128,

## Table 8, empirical matrices

For every sample the five propagators use the same evaluations; the energy form is $\widehat{\mathbf J}^{\mathrm{en}}_m=m^{-1}(\mathcal A\boldsymbol\Psi)^*(\mathcal A\boldsymbol\Psi)$.
The perturbations are operator norms on $\mathcal V_N$.

In [4]:
def empirical(x, N, chunk=100_000):
    """Empirical G, H, J (pointwise), J (energy) from the samples x, accumulated in chunks."""
    G, H, Jp, Je = (np.zeros((N, N)) for _ in range(4))
    for s in range(0, x.size, chunk):
        P, AP, A2P = arrays(x[s:s + chunk], N)
        G += P.T @ P; H += P.T @ AP; Jp += P.T @ A2P; Je += AP.T @ AP
    return G / x.size, H / x.size, Jp / x.size, Je / x.size


rng = np.random.default_rng(SEED)
REPS = 50
print(f"errors ||P Pi_N f - K^T f|| with empirical matrices, means over {REPS} repetitions")
print("    N        m   " + "  ".join(f"{name:>18s}" for name in NAMES) + "    |Ahat-A|  |Bhat-B| (energy)")
sampled = []
for N in [16, 32]:
    M = Model(N)
    for m in [10**4, 10**5, 10**6]:
        errs, dev = np.zeros((REPS, 5)), np.zeros((REPS, 2))
        for r in range(REPS):
            G, H, Jp, Je = empirical(rng.uniform(0.0, 2 * np.pi, m), N)
            Ah, Bp, Be = np.linalg.solve(G, H), np.linalg.solve(G, Jp), np.linalg.solve(G, Je)
            props, _ = propagators(Ah, Bp, Be, M.dt, M.n)
            errs[r] = [M.error(P @ M.c0) for P in props]
            dev[r] = [M.op_norm(Ah - M.A), M.op_norm(Be - M.Be)]
        sampled.append(dict(N=N, m=m, err=errs.mean(axis=0), lo=np.percentile(errs, 10, axis=0), hi=np.percentile(errs, 90, axis=0),
                            dev=dev.mean(axis=0), r_beats_energy=float(np.mean(errs[:, 2] < errs[:, 4]))))
        print(f"{N:5d} {m:8d}   " + "  ".join(f"{e:18.3e}" for e in errs.mean(axis=0)) + f"    {dev[:, 0].mean():.2e}  {dev[:, 1].mean():.2e}")
print("10th and 90th percentiles of the errors of SDMD-R and of SDMD-2 (energy), and share of the repetitions in which SDMD-R is the more accurate:")
for r in sampled:
    print(f"  N = {r['N']:3d}, m = {r['m']:7d}: SDMD-R [{r['lo'][2]:.2e}, {r['hi'][2]:.2e}], SDMD-2 (energy) [{r['lo'][4]:.2e}, {r['hi'][4]:.2e}], {100 * r['r_beats_energy']:.0f}%")
ex = {r["N"]: r["err"] for r in exact}
print("mean error divided by the error of the same propagator with exact matrices:")
for r in sampled:
    print(f"  N = {r['N']:3d}, m = {r['m']:7d}: " + ", ".join(f"{name} {r['err'][j] / ex[r['N']][j]:.3g}" for j, name in enumerate(NAMES)))
print("mean error of SDMD-2 (energy) divided by the mean error of SDMD-R:",
      ", ".join(f"{r['err'][4] / r['err'][2]:.2f}" for r in sampled))
print("mean error of SDMD-R divided by the mean error of the exponential:",
      ", ".join(f"{r['err'][2] / r['err'][0]:.3f}" for r in sampled))

errors ||P Pi_N f - K^T f|| with empirical matrices, means over 50 repetitions
    N        m          exponential              SDMD-1              SDMD-R  SDMD-2 (pointwise)     SDMD-2 (energy)    |Ahat-A|  |Bhat-B| (energy)


   16    10000            1.363e-05           3.872e-04           1.396e-05           5.425e-04           6.555e-05    8.28e-01  1.97e+02


   16   100000            8.425e-06           3.868e-04           8.953e-06           5.426e-04           1.383e-05    2.61e-01  6.24e+01


   16  1000000            7.935e-06           3.866e-04           8.497e-06           5.426e-04           8.873e-06    8.45e-02  1.94e+01


   32    10000            2.306e-06           8.897e-05           2.320e-06           1.236e-04           7.355e-05    4.60e+00  5.39e+03


   32   100000            7.905e-07           8.927e-05           7.960e-07           1.239e-04           8.447e-06    1.47e+00  1.70e+03


   32  1000000            3.952e-07           8.915e-05           4.234e-07           1.239e-04           1.251e-06    4.39e-01  5.48e+02
10th and 90th percentiles of the errors of SDMD-R and of SDMD-2 (energy), and share of the repetitions in which SDMD-R is the more accurate:
  N =  16, m =   10000: SDMD-R [1.00e-05, 1.94e-05], SDMD-2 (energy) [2.99e-05, 1.16e-04], 100%
  N =  16, m =  100000: SDMD-R [8.47e-06, 9.59e-06], SDMD-2 (energy) [1.02e-05, 1.67e-05], 100%
  N =  16, m = 1000000: SDMD-R [8.39e-06, 8.63e-06], SDMD-2 (energy) [8.56e-06, 9.14e-06], 98%
  N =  32, m =   10000: SDMD-R [1.26e-06, 3.59e-06], SDMD-2 (energy) [3.01e-05, 1.55e-04], 100%
  N =  32, m =  100000: SDMD-R [5.47e-07, 1.14e-06], SDMD-2 (energy) [3.92e-06, 1.40e-05], 100%
  N =  32, m = 1000000: SDMD-R [3.66e-07, 5.01e-07], SDMD-2 (energy) [8.05e-07, 1.94e-06], 100%
mean error divided by the error of the same propagator with exact matrices:
  N =  16, m =   10000: exponential 1.73, SDMD-1 1, SDMD-R 1.65, SDMD-

## LaTeX rows of Table 8

In [5]:
def sci(x, sig=3):
    mant, ex_ = f"{x:.{sig-1}e}".split("e")
    return f"${mant}\\times10^{{{int(ex_)}}}$"


for r in exact:
    print(" & ".join([str(r["N"]), "exact", str(r["n"])] + [sci(e) for e in r["err"]]) + r" \\")
for r in sampled:
    n = next(e["n"] for e in exact if e["N"] == r["N"])
    print(" & ".join([str(r["N"]), f"$10^{{{int(round(np.log10(r['m'])))}}}$", str(n)] + [sci(e) for e in r["err"]]) + r" \\")

16 & exact & 54 & $7.87\times10^{-6}$ & $3.87\times10^{-4}$ & $8.45\times10^{-6}$ & $5.43\times10^{-4}$ & $8.37\times10^{-6}$ \\
32 & exact & 233 & $3.22\times10^{-7}$ & $8.91\times10^{-5}$ & $3.58\times10^{-7}$ & $1.24\times10^{-4}$ & $3.57\times10^{-7}$ \\
64 & exact & 977 & $1.80\times10^{-8}$ & $2.12\times10^{-5}$ & $2.00\times10^{-8}$ & $2.95\times10^{-5}$ & $2.00\times10^{-8}$ \\
128 & exact & 4002 & $1.09\times10^{-9}$ & $5.18\times10^{-6}$ & $1.21\times10^{-9}$ & $7.18\times10^{-6}$ & $1.21\times10^{-9}$ \\
16 & $10^{4}$ & 54 & $1.36\times10^{-5}$ & $3.87\times10^{-4}$ & $1.40\times10^{-5}$ & $5.42\times10^{-4}$ & $6.56\times10^{-5}$ \\
16 & $10^{5}$ & 54 & $8.43\times10^{-6}$ & $3.87\times10^{-4}$ & $8.95\times10^{-6}$ & $5.43\times10^{-4}$ & $1.38\times10^{-5}$ \\
16 & $10^{6}$ & 54 & $7.94\times10^{-6}$ & $3.87\times10^{-4}$ & $8.50\times10^{-6}$ & $5.43\times10^{-4}$ & $8.87\times10^{-6}$ \\
32 & $10^{4}$ & 233 & $2.31\times10^{-6}$ & $8.90\times10^{-5}$ & $2.32\times10^{-6

## Software versions of this run

In [6]:
import platform
import scipy

print("Python", platform.python_version(), "| NumPy", np.__version__, "| SciPy", scipy.__version__)

Python 3.13.18 | NumPy 2.5.3 | SciPy 1.18.1
